In [1]:
from collections import Counter, defaultdict
import json
from pathlib import Path

# Configuration from your setup
STUDENT_ID = "TUPM-26-4014"
SEED_DIGIT = int(STUDENT_ID[-1])  # 4
TARGET_YEAR = str(2015 + SEED_DIGIT)  # '2019'

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
BACKUP_DIR = Path("backup")

OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

csv_path = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"
filtered_csv_path = OUTPUT_DIR / f"{STUDENT_ID}_cyber_filtered.csv"
json_summary_path = OUTPUT_DIR / f"{STUDENT_ID}_cyber_summary.json"
backup_json_path = BACKUP_DIR / f"{STUDENT_ID}_cyber_summary.bak"
report_file_path = OUTPUT_DIR / f"{STUDENT_ID}_cyber_report.txt"

total_records = 0
target_year_count = 0
attack_type_counts = Counter()
total_loss = 0.0
target_incidents = []

# Define standard headers for fallback and export
DEFAULT_HEADER = [
    "Date",
    "Attack Type",
    "Financial Loss (in Million $)",
    "Incident Details",
]

try:
  with open(csv_path, "r", encoding="utf-8") as f:
    first_line = f.readline().strip()
    first_parts = [p.strip() for p in first_line.split(",")]

    has_header = False
    if len(first_parts) >= 3:
      # If the 3rd element (loss) is not numeric, it's a header
      if not first_parts[2].replace(".", "", 1).isdigit():
        has_header = True

    # If no header, process the first line as a data row
    if not has_header and len(first_parts) >= 3:
      total_records += 1
      date_str = first_parts[0]
      if TARGET_YEAR in date_str:
        target_year_count += 1
        attack_type = first_parts[1]
        attack_type_counts[attack_type] += 1
        try:
          loss = float(first_parts[2])
        except ValueError:
          loss = 0.0
        total_loss += loss
        target_incidents.append(
            {"row": first_parts, "loss": loss, "attack_type": attack_type}
        )

    # Process remaining lines line-by-line
    for line in f:
      parts = [p.strip() for p in line.strip().split(",")]
      if len(parts) >= 3:
        total_records += 1
        date_str = parts[0]

        # Check if target year occurs in the date string
        if TARGET_YEAR in date_str:
          target_year_count += 1
          attack_type = parts[1]
          attack_type_counts[attack_type] += 1

          try:
            loss = float(parts[2])
          except ValueError:
            loss = 0.0  # Handle invalid financial values safely

          total_loss += loss
          target_incidents.append(
              {
                  "row": parts,
                  "loss": loss,
                  "attack_type": attack_type,
              }
          )

except FileNotFoundError:
  print(f"Error: {csv_path} not found. Please place it in the 'data/' directory.")

avg_loss = (total_loss / target_year_count) if target_year_count > 0 else 0.0

# Sort incidents by descending financial loss to find costliest ones
target_incidents.sort(key=lambda x: x["loss"], reverse=True)
five_costliest = target_incidents[:5]

# Export selected incidents to filtered CSV
with open(filtered_csv_path, "w", encoding="utf-8") as out_f:
  out_f.write(",".join(DEFAULT_HEADER) + "\n")
  for inc in target_incidents:
    out_f.write(",".join(inc["row"]) + "\n")

# Write summary JSON
summary_data = {
    "target_year": TARGET_YEAR,
    "total_records": total_records,
    "incidents_in_target_year": target_year_count,
    "count_per_attack_type": dict(attack_type_counts),
    "total_financial_loss_million_usd": round(total_loss, 2),
    "average_financial_loss_million_usd": round(avg_loss, 2),
    "five_costliest_incidents": [
        {"loss": inc["loss"], "type": inc["attack_type"]} for inc in five_costliest
    ],
}

with open(json_summary_path, "w", encoding="utf-8") as json_f:
  json.dump(summary_data, json_f, indent=4)

# Create backup of JSON summary
import shutil

shutil.copy(json_summary_path, backup_json_path)
backup_verified = backup_json_path.exists()

# Generate text report file
with open(report_file_path, "w", encoding="utf-8") as rep_f:
  rep_f.write(f"=== CYBERSECURITY INCIDENT REPORT ({TARGET_YEAR}) ===\n")
  rep_f.write(f"Student ID: {STUDENT_ID}\n")
  rep_f.write(f"Total Incidents in {TARGET_YEAR}: {target_year_count}\n")
  rep_f.write(f"Total Financial Loss: ${total_loss:,.2f} Million\n")
  rep_f.write(f"Average Financial Loss: ${avg_loss:,.2f} Million\n\n")
  rep_f.write("Attack Type Breakdown:\n")
  for atype, count in attack_type_counts.items():
    rep_f.write(f" - {atype}: {count}\n")
  rep_f.write("\nFive Costliest Incidents:\n")
  for i, inc in enumerate(five_costliest, 1):
    rep_f.write(
        f" {i}. Loss: ${inc['loss']}M | Type: {inc['attack_type']}\n"
    )

print("--- Exercise 3 Assessment Data (Header-Safe) ---")
print(f"Target year: {TARGET_YEAR}")
print(f"Total records in dataset: {total_records}")
print(f"Incidents in target year: {target_year_count}")
print(f"Count per attack type: {dict(attack_type_counts)}")
print(f"Total financial loss (USD millions): ${total_loss:,.2f}M")
print(f"Average financial loss (USD millions): ${avg_loss:,.2f}M")
print(
    "Five costliest incidents:"
    f" {[inc['loss'] for inc in five_costliest]}"
)
print(f"Filtered CSV file: {filtered_csv_path}")
print(f"JSON summary file: {json_summary_path}")
print(f"Backup verification: {backup_verified}")
print(f"Report file: {report_file_path}")

Error: data\Global_Cybersecurity_Threats_2015-2024.csv not found. Please place it in the 'data/' directory.
--- Exercise 3 Assessment Data (Header-Safe) ---
Target year: 2019
Total records in dataset: 0
Incidents in target year: 0
Count per attack type: {}
Total financial loss (USD millions): $0.00M
Average financial loss (USD millions): $0.00M
Five costliest incidents: []
Filtered CSV file: output\TUPM-26-4014_cyber_filtered.csv
JSON summary file: output\TUPM-26-4014_cyber_summary.json
Backup verification: True
Report file: output\TUPM-26-4014_cyber_report.txt
